In [50]:
# Import necessary libraries
import pandas as pd
import geopandas as gpd
from sklearn.cluster import KMeans
import folium

In [51]:
# Step 1: Load GeoJSON Data
geojson_path = "./voivodships.geojson"  # Ensure this file exists in your working directory
try:
   polygons = gpd.read_file(geojson_path)
except FileNotFoundError:
   raise FileNotFoundError(f"GeoJSON file not found: {geojson_path}")


# Debug: Print available columns in the GeoJSON file
print("Columns in GeoJSON file:", polygons.columns)

Columns in GeoJSON file: Index(['CCA_1', 'CCN_1', 'ENGTYPE_1', 'HASC_1', 'ID_0', 'ID_1', 'ISO',
       'NAME_0', 'NAME_1', 'NL_NAME_1', 'TYPE_1', 'name', 'geometry'],
      dtype='object')


In [52]:
# Step 2: Add Population Data
population_data = pd.DataFrame({
   'voivodeship': ["Dolnoslaskie", "Kujawsko-pomorskie", "Lubelskie", "Lubuskie",
                   "Lodzkie", "Malopolskie", "Mazowieckie", "Opolskie",
                   "Podkarpackie", "Podlaskie", "Pomorskie", "Slaskie",
                   "Swietokrzyskie", "Warminsko-mazurskie", "Wielkopolskie",
                   "Zachodniopomorskie"],
   'population': [2874496, 1990323, 2003475, 972140, 2354135, 3429685,
                  5510618, 933349, 2067139, 1135201, 2359956, 4305126,
                  1163001, 1353374, 3484177, 1626876],
   'num_cities': [93, 56, 57, 44, 60, 64, 111, 37, 54, 40, 43, 74, 51, 50, 120, 66]
})


# Align GeoJSON column names to match population data
# Debug: Check unique names in the GeoJSON file
print("Unique names in NAME_1 column:", polygons['NAME_1'].unique())

Unique names in NAME_1 column: ['Łódź' 'Świętokrzyskie' 'Greater Poland' 'Kuyavian-Pomeranian'
 'Lesser Poland' 'Lower Silesian' 'Lublin' 'Lubusz' 'Masovian' 'Opole'
 'Podlachian' 'Pomeranian' 'Silesian' 'Subcarpathian' 'Warmian-Masurian'
 'West Pomeranian']


In [53]:
# Step 3: Map GeoJSON Names to Population Data Names
# Complete mapping between GeoJSON names and voivodeship names
name_mapping = {
   "Lower Silesian": "Dolnoslaskie",
   "Kuyavian-Pomeranian": "Kujawsko-pomorskie",
   "Lublin": "Lubelskie",
   "Lubusz": "Lubuskie",
   "Łódź": "Lodzkie",
   "Lesser Poland": "Malopolskie",
   "Masovian": "Mazowieckie",
   "Opole": "Opolskie",
   "Subcarpathian": "Podkarpackie",
   "Podlachian": "Podlaskie",
   "Pomeranian": "Pomorskie",
   "Silesian": "Slaskie",
   "Świętokrzyskie": "Swietokrzyskie",
   "Warmian-Masurian": "Warminsko-mazurskie",
   "Greater Poland": "Wielkopolskie",
   "West Pomeranian": "Zachodniopomorskie"
}


# Replace GeoJSON names with mapped names
polygons['NAME_1'] = polygons['NAME_1'].replace(name_mapping)


# Merge GeoJSON data with population data
polygons = polygons.merge(population_data, left_on="NAME_1", right_on="voivodeship", how="inner")


# Debug: Check the merged DataFrame
print("Number of rows after merge:", len(polygons))
print(polygons[['NAME_1', 'voivodeship', 'population', 'num_cities']])

Number of rows after merge: 16
                 NAME_1          voivodeship  population  num_cities
0               Lodzkie              Lodzkie     2354135          60
1        Swietokrzyskie       Swietokrzyskie     1163001          51
2         Wielkopolskie        Wielkopolskie     3484177         120
3    Kujawsko-pomorskie   Kujawsko-pomorskie     1990323          56
4           Malopolskie          Malopolskie     3429685          64
5          Dolnoslaskie         Dolnoslaskie     2874496          93
6             Lubelskie            Lubelskie     2003475          57
7              Lubuskie             Lubuskie      972140          44
8           Mazowieckie          Mazowieckie     5510618         111
9              Opolskie             Opolskie      933349          37
10            Podlaskie            Podlaskie     1135201          40
11            Pomorskie            Pomorskie     2359956          43
12              Slaskie              Slaskie     4305126          74
13 

In [54]:
# Step 4: Handle Missing Values
# Check and fill missing values in 'population' if needed
if polygons['population'].isna().sum() > 0:
   print("Missing values found in 'population'. Filling with the column mean...")
   polygons['population'] = polygons['population'].fillna(polygons['population'].mean())


# Standardize the population data for clustering
polygons['population_scaled'] = (
   polygons['population'] - polygons['population'].mean()
) / polygons['population'].std()


# Handle any potential NaN values in 'population_scaled'
polygons['population_scaled'] = polygons['population_scaled'].fillna(0)

In [55]:
# Step 5: Apply K-Means Clustering
# Ensure there are enough rows for clustering
if len(polygons) < 3:
   raise ValueError("Not enough data points for K-Means clustering (minimum 3 required).")


kmeans = KMeans(n_clusters=3, random_state=42)
polygons['cluster'] = kmeans.fit_predict(polygons[['population_scaled']])

# Step 5b: Sort clusters by population and reassign labels
cluster_order = polygons.groupby('cluster')['population'].mean().sort_values().index
cluster_mapping = {old: new for new, old in enumerate(cluster_order)}

# Apply the sorted cluster mapping
polygons['cluster'] = polygons['cluster'].map(cluster_mapping)


# Debug: Check the clustering results
print(polygons[['voivodeship', 'population', 'cluster']])

            voivodeship  population  cluster
0               Lodzkie     2354135        1
1        Swietokrzyskie     1163001        0
2         Wielkopolskie     3484177        1
3    Kujawsko-pomorskie     1990323        0
4           Malopolskie     3429685        1
5          Dolnoslaskie     2874496        1
6             Lubelskie     2003475        0
7              Lubuskie      972140        0
8           Mazowieckie     5510618        2
9              Opolskie      933349        0
10            Podlaskie     1135201        0
11            Pomorskie     2359956        1
12              Slaskie     4305126        2
13         Podkarpackie     2067139        0
14  Warminsko-mazurskie     1353374        0
15   Zachodniopomorskie     1626876        0


In [56]:
# Step 6: Create an Interactive Choropleth Map
# Initialize the map
m = folium.Map(location=[52, 19], zoom_start=6)  # Centered on Poland


# Add a choropleth layer
folium.Choropleth(
   geo_data=polygons,  # GeoJSON data
   data=polygons,  # DataFrame with cluster info
   columns=['voivodeship', 'cluster'],  # Columns for the choropleth
   key_on='feature.properties.NAME_1',  # Match GeoJSON property
   fill_color='YlGn',
   fill_opacity=0.7,
   line_opacity=0.2,
   legend_name='Population Clusters'
).add_to(m)


# Add pop-ups for each voivodeship
for _, row in polygons.iterrows():
   # Extract the centroid of each geometry for placing the marker
   centroid = row.geometry.centroid
   folium.Marker(
       location=[centroid.y, centroid.x],
       popup=folium.Popup(
           f"<b>Voivodeship:</b> {row['voivodeship']}<br>"
           f"<b>Population:</b> {row['population']:,}<br>"
           f"<b>Number of Cities:</b> {row['num_cities']}<br>",
           max_width=300
       )
   ).add_to(m)


# Save the map
output_path = "poland_population_clusters_geojson.html"
m.save(output_path)
print(f"Interactive map saved as '{output_path}'")

Interactive map saved as 'poland_population_clusters_geojson.html'
